# Block 1 — Document Normalization & Batch ROI Extraction
**Medical Document Intelligence System**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RwaRwa599/epq3/blob/block2/block1/Block_1_Document_Normalization.ipynb)

---
### Pipeline Overview
Block 1 rectifies and extracts all ROI crops from medical lab request sheets (scans and phone camera photos):
1. **Quad Detection & Perspective Warp:** Detects page quadrilateral and maps to canonical canvas (`2048×1754` or `2048×1720`).
2. **Orientation Correction:** Ensures header sits at top (0° or 180° rotation).
3. **Fine Alignment & 1-to-1 Snapping:** Aligns landmarks and column gutters.
4. **Normalized ROI Extraction:** Extracts 138 checkboxes + 12 handwriting fields per document.
5. **Batch ZIP Packaging:** Packs all normalized assets into `block1_normalized_batch.zip` for direct ingestion into **Block 2**.

## 1. Setup Environment
Install dependencies (`opencv-python-headless`, `Pillow`, `pydantic`, `matplotlib`) and import `med_doc`.

In [ ]:
import os, sys, glob

# In Google Colab, clone repo if running in a fresh session:
if not os.path.exists('med_doc'):
    !git clone -b block2 https://github.com/RwaRwa599/epq3.git repo
    if os.path.exists('repo/block1'):
        sys.path.insert(0, 'repo/block1')
        os.chdir('repo/block1')
    else:
        sys.path.insert(0, 'repo')
        os.chdir('repo')

!pip install -q "opencv-python-headless>=4.8.0" "Pillow>=10.0.0" "pydantic>=2.0.0" "matplotlib>=3.7.0"

from med_doc.normalization import normalize_document, normalize_batch
print("✓ Environment initialized and med_doc imported successfully!")

## 2. Upload Document Images (Batch Mode)
Upload multiple images or a `.zip` of images via the Colab upload popup. If no files are uploaded, default sample images will be used.

In [ ]:
upload_dir = 'input_batch'
os.makedirs(upload_dir, exist_ok=True)

try:
    from google.colab import files
    print("Choose image files (.png, .jpg, .jpeg) or a .zip file from your computer:")
    uploaded = files.upload()
    for filename in uploaded.keys():
        os.rename(filename, os.path.join(upload_dir, filename))
        print(f"Uploaded: {filename}")
except Exception as e:
    print("Not running in interactive Colab upload or upload skipped. Using existing samples.")

# Check uploaded or sample files
batch_inputs = glob.glob(f"{upload_dir}/*.*")
if not batch_inputs:
    batch_inputs = glob.glob("samples/*.*")

print(f"\nTotal input document(s) queued for batch processing: {len(batch_inputs)}")
for p in batch_inputs:
    print(f"  • {p}")

## 3. Run Batch Normalization & Extract Crops
Processes all queued documents and packages them into `block1_normalized_batch.zip`.

In [ ]:
output_zip_path = "block1_normalized_batch.zip"
output_dir = "outputs/block1_batch"

result = normalize_batch(
    batch_inputs,
    output_dir=output_dir,
    output_zip=output_zip_path
)

manifest = result["manifest"]
print(f"\n[+] Batch Processing Summary:")
print(f"    Total: {manifest['total_documents']} | Success: {manifest['successful_documents']}")
for doc in manifest["documents"]:
    print(f"    - {doc['doc_id']}: {doc['num_checkboxes']} checkboxes, {doc['num_handwriting']} handwriting fields")

## 4. Visual Inspection of First Document
Inspect the rectified canonical canvas, fine-aligned visual overlay, and sample checkbox/handwriting crops.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

first_doc_id = manifest["documents"][0]["doc_id"]
overlay_img_path = f"{output_dir}/docs/{first_doc_id}/overlay.png"

if os.path.exists(overlay_img_path):
    plt.figure(figsize=(14, 10))
    plt.imshow(Image.open(overlay_img_path))
    plt.title(f"Fine-Aligned Visual Overlay: {first_doc_id}", fontsize=14)
    plt.axis('off')
    plt.show()

# Display sample checkbox and handwriting crops
sample_crops = glob.glob(f"{output_dir}/docs/{first_doc_id}/crops/*/*.png")[:8]
if sample_crops:
    fig, axes = plt.subplots(2, 4, figsize=(16, 6))
    for ax, crop_p in zip(axes.flat, sample_crops):
        ax.imshow(Image.open(crop_p))
        ax.set_title(os.path.basename(crop_p), fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## 5. Download `block1_normalized_batch.zip` for Block 2
Download the generated ZIP file containing all normalized canonical sheets, crop patches, and metadata. This file can be uploaded directly into **Block 2**!

In [ ]:
try:
    from google.colab import files
    print(f"Downloading {output_zip_path} to your computer...")
    files.download(output_zip_path)
except Exception as e:
    print(f"ZIP file ready locally at: {os.path.abspath(output_zip_path)}")